In [ ]:
# 데이터 경로 설정 (예시 경로 수정 필요)
data_folder = r"C:\Users\\705-2\\dataset_file\\fan\train"
wav_files = [file for file in os.listdir(data_folder) if file.endswith('.wav')]

# 임의로 3개의 파일 선택
selected_files = wav_files[:3]

def visualize_audio(file_path):
    # 오디오 파일 로드
    y, sr = librosa.load(file_path, sr=16000)

    # 시각화
    plt.figure(figsize=(14, 8))

    # 파형 시각화
    plt.subplot(2, 1, 1)
    librosa.display.waveshow(y, sr=sr)
    plt.title(f'Waveform of {file_path}')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')

    # 스펙트로그램 시각화
    plt.subplot(2, 1, 2)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=1024, hop_length=512, n_mels=128)
    S_dB = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_dB, sr=sr, hop_length=512, x_axis='time', y_axis='mel')
    plt.title(f'Spectrogram of {file_path}')
    plt.colorbar(format='%+2.0f dB')
    plt.xlabel('Time (s)')
    plt.ylabel('Frequency (Hz)')

    plt.tight_layout()
    plt.show()

# 선택한 3개의 파일에 대해 시각화 수행
for file_name in selected_files:
    file_path = os.path.join(data_folder, file_name)
    visualize_audio(file_path)

import os
import pandas as pd
import librosa
import numpy as np
from scipy.signal import butter, lfilter
from tqdm import tqdm

# 저주파 필터링 함수 정의
def apply_bandpass_filter(audio, sr, low_cut=50, high_cut=2000, order=5):
    nyquist = 0.5 * sr
    low = low_cut / nyquist
    high = (high_cut - 1) / nyquist
    b, a = butter(order, [low, high], btype='band')
    filtered_audio = lfilter(b, a, audio)
    return filtered_audio

# 데이터 증강 함수 정의
def augment_audio(data, sr):
    # 시간 스트레칭 (0.8~1.2배)
    stretched = librosa.effects.time_stretch(data, rate=np.random.uniform(0.8, 1.2))
    stretched = np.clip(stretched, -1.0, 1.0)

    # 피치 시프트 (-2~+2 반음)
    pitched = librosa.effects.pitch_shift(data, sr=sr, n_steps=np.random.uniform(-2, 2))
    pitched = np.clip(pitched, -1.0, 1.0)

    # 백색 잡음 추가
    noise = np.random.randn(len(data)) * 0.005
    noisy = data + noise
    noisy = np.clip(noisy, -1.0, 1.0)

    return [stretched, pitched, noisy]

train_folders = [
        r"C:\Users\\705-2\\dataset_file\\fan\train",
        r"C:\Users\\705-2\\dataset_file\\pump\train",
        r"C:\Users\\705-2\\dataset_file\\slider\train",
        r"C:\Users\\705-2\\dataset_file\\valve\train"
]

# 데이터를 저장할 빈 리스트
train_data = []

# 학습 데이터 처리
for folder in tqdm(train_folders):
    folder_path = folder
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.wav'):
            file_path = os.path.join(folder_path, file_name)

            # wav 파일 불러오기
            y, sr = librosa.load(file_path, sr=16000)

            # 원본 오디오 및 증강된 오디오 배열 생성
            augmented_audios = [y] + augment_audio(y, sr)

            # 각 오디오 파일에 대해 특징 추출
            for audio_data in augmented_audios:
                # 대역 필터링 적용
                filtered_audio = apply_bandpass_filter(audio_data, lowcut=50, highcut=2000, fs=sr)

                # MFCC 추출
                mfccs = librosa.feature.mfcc(y=filtered_audio, sr=sr, n_mfcc=64, n_fft=2024, hop_length=512)
                mfccs_mean = mfccs.mean(axis=1)  # 각 MFCC의 평균값

                # Mel Spectrogram 추출
                mel_spectrogram = librosa.feature.melspectrogram(y=filtered_audio, sr=sr, n_mels=128, n_fft=2024, hop_length=512)
                mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)  # dB로 변환
                mel_spectrogram_mean = mel_spectrogram_db.mean(axis=1)  # Mel Spectrogram의 평균값

                # 데이터 저장 (파일명과 폴더명도 저장하여 구분)
                train_data.append({
                    'file_name': file_name,
                    'folder': folder,
                    'mfcc': mfccs_mean,
                    'mel_spectrogram': mel_spectrogram_mean
                })

# DataFrame 생성
train_df = pd.DataFrame(train_data)

test_folders = [
    r"C:\Users\\705-2\\dataset_file\\fan\\test",
    r"C:\Users\\705-2\\dataset_file\\pump\\test",
    r"C:\Users\\705-2\\dataset_file\\slider\\test",
    r"C:\Users\\705-2\\dataset_file\\valve\\test"
]

test_data = []

for folder in tqdm(test_folders):
    folder_path = folder
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.wav'):
            file_path = os.path.join(folder_path, file_name)

            # Load the wav file
            y, sr = librosa.load(file_path, sr=16000)

            # Apply bandpass filtering
            filtered_audio = apply_bandpass_filter(y, lowcut=50, highcut=2000, fs=sr)

            # Extract MFCC features
            mfccs = librosa.feature.mfcc(y=filtered_audio, sr=sr, n_mfcc=64, n_fft=2024, hop_length=512)
            mfccs_mean = mfccs.mean(axis=1)  # Mean of each MFCC feature

            # Extract Mel Spectrogram
            mel_spectrogram = librosa.feature.melspectrogram(y=filtered_audio, sr=sr, n_mels=128, n_fft=2024, hop_length=512)
            mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)
            mel_spectrogram_mean = mel_spectrogram_db.mean(axis=1)


            test_data.append({
                'file_name': file_name,
                'folder': folder,
                'mfcc': mfccs_mean,
                'mel_spectrogram': mel_spectrogram_mean
            })

# Create DataFrame
test_df = pd.DataFrame(test_data)

import re

# 학습 데이터에 'id' 열 추가
train_df['id'] = train_df['file_name'].apply(lambda x: int(re.search(r'id_(\d+)', x).group(1)) if re.search(r'id_(\d+)', x) else None)

# 학습 데이터에 'equipment' 열 추가
train_df['equipment'] = train_df['folder'].apply(lambda x: re.search(r'\\(\w+)\\train', x).group(1) if re.search(r'\\(\w+)\\train', x) else None)

# 테스트 데이터에 'id' 열 추가
test_df['id'] = test_df['file_name'].apply(lambda x: int(re.search(r'id_(\d+)', x).group(1)) if re.search(r'id_(\d+)', x) else None)

# 테스트 데이터에 'equipment' 열 추가
test_df['equipment'] = test_df['folder'].apply(lambda x: x.split(os.path.sep)[-3] if 'test' in x.split(os.path.sep) else None)

from sklearn.preprocessing import MinMaxScaler
import numpy as np

# MinMaxScaler 초기화
scaler_mfcc = MinMaxScaler()
scaler_mel = MinMaxScaler()

# 학습 셋에서 스케일링 적용 (mfcc와 mel_spectrogram)
train_df['mfcc_scaled'] = list(scaler_mfcc.fit_transform(np.array(train_df['mfcc'].tolist())))
train_df['mel_spectrogram_scaled'] = list(scaler_mel.fit_transform(np.array(train_df['mel_spectrogram'].tolist())))

# 테스트 셋에서 스케일링 적용 (학습 셋에서 학습한 scaler 사용)
test_df['mfcc_scaled'] = list(scaler_mfcc.transform(np.array(test_df['mfcc'].tolist())))
test_df['mel_spectrogram_scaled'] = list(scaler_mel.transform(np.array(test_df['mel_spectrogram'].tolist())))

# train_df와 test_df에서 불필요한 열 삭제
train_df = train_df.drop(columns=['mfcc', 'mel_spectrogram', 'folder'])
test_df = test_df.drop(columns=['mfcc', 'mel_spectrogram', 'folder'])

# train_df와 test_df에 'label' 열 추가
train_df['label'] = train_df['file_name'].apply(lambda x: 0 if 'normal' in x else 1)
test_df['label'] = test_df['file_name'].apply(lambda x: 0 if 'normal' in x else 1)

# 필요한 패키지 불러오기
from sklearn.model_selection import train_test_split

# 그룹화하여 장비 유형과 ID별로 분리 및 모델 학습 준비
for (equipment_type, equipment_id), group_data in train_df.groupby(['equipment', 'id']):
    print(f"Equipment: {equipment_type}, ID: {equipment_id}")

    # 특징 데이터와 레이블 분리 (예: mfcc_scaled, mel_spectrogram_scaled)
    X = group_data[['mfcc_scaled', 'mel_spectrogram_scaled']]
    y = group_data['label'] if 'label' in group_data.columns else None  # 레이블이 있는 경우만

    # 데이터셋을 학습과 검증 세트로 나누기 (레이블이 있는 경우에만)
    if y is not None:
        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

        # 여기에서 모델 정의 및 학습 시작
        # 예시: model.fit(X_train, y_train)

    else:
        # 레이블이 없을 경우 전체 데이터로 모델 학습
        X_train = X

        # 여기에서 모델 정의 및 학습 시작
        # 예시: model.fit(X_train)

    # 각 모델 학습이 끝난 후 모델 저장 혹은 평가 코드 작성 가능
    print(f"Model trained for {equipment_type} ID {equipment_id}\n")

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
import joblib

# 장비별 id => LSTM_autoencoder 모델 정의 함수
def create_lstm_autoencoder(input_shape):
    inputs = Input(shape=input_shape)

    # 인코딩 부분
    x = LSTM(256, activation="tanh", recurrent_activation="sigmoid", return_sequences=True)(inputs)
    x = LSTM(128, activation="tanh", recurrent_activation="sigmoid", return_sequences=True)(x)
    x = LSTM(64, activation="tanh", recurrent_activation="sigmoid", return_sequences=False)(x)
    encoded = Dense(32, activation="relu")(x)  # 최종 인코딩 레이어

    # 디코딩 부분
    x = RepeatVector(input_shape[0])(encoded)
    x = LSTM(64, activation="tanh", recurrent_activation="sigmoid", return_sequences=True)(x)
    x = LSTM(128, activation="tanh", recurrent_activation="sigmoid", return_sequences=True)(x)
    x = LSTM(256, activation="tanh", recurrent_activation="sigmoid", return_sequences=True)(x)
    decoded = TimeDistributed(Dense(input_shape[1]))(x)  # 원래 차원으로 복원

    # 모델 정의 및 컴파일
    autoencoder = Model(inputs, decoded)
    autoencoder.compile(optimizer="adam", loss="mse")

    return autoencoder

# 그룹화하여 장비 유형과 ID별로 모델 학습 준비(mfcc_scaled) - cuDNN사용
for (equipment_type, equipment_id), group_data in train_df.groupby(['equipment', 'id']):
    print(f"Training models for Equipment: {equipment_type}, ID: {equipment_id}")

    # 1. MFCC 특징에 대해 LSTM Autoencoder 모델 학습
    X_mfcc = np.array(group_data['mfcc_scaled'].tolist())
    X_mfcc = X_mfcc.reshape((X_mfcc.shape[0], X_mfcc.shape[1], 1))  # (samples, time_steps, 1)

    # 데이터셋을 학습과 검증 세트로 나누기
    X_train, X_val = train_test_split(X_mfcc, test_size=0.2, random_state=42)

    # LSTM Autoencoder 모델 생성 및 학습 (MFCC)
    input_shape = (X_train.shape[1], X_train.shape[2])
    lstm_autoencoder_mfcc = create_lstm_autoencoder(input_shape)

    # EarlyStopping 콜백 설정
    early_stopping = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

    # 모델 학습 (MFCC)
    lstm_autoencoder_mfcc.fit(
        X_train, X_train,
        epochs=100,
        batch_size=32,
        validation_data=(X_val, X_val),
        callbacks=[early_stopping]
    )

    # 학습된 MFCC 모델 저장
    mfcc_model_filename = f"{equipment_type}_{equipment_id}_mfcc_lstm_autoencoder.h5"
    lstm_autoencoder_mfcc.save(mfcc_model_filename)
    print(f"MFCC LSTM Autoencoder model saved as {mfcc_model_filename}\n")


# 2. Mel Spectrogram 특징에 대해 LSTM Autoencoder 모델 학습
for (equipment_type, equipment_id), group_data in train_df.groupby(['equipment', 'id']):
    print(f"Training models for Equipment: {equipment_type}, ID: {equipment_id}")

    X_mel = np.array(group_data['mel_spectrogram_scaled'].tolist())
    X_mel = X_mel.reshape((X_mel.shape[0], X_mel.shape[1], 1))  # (samples, time_steps, 1)

    # 데이터셋을 학습과 검증 세트로 나누기
    X_train, X_val = train_test_split(X_mel, test_size=0.2, random_state=42)

    # LSTM Autoencoder 모델 생성 및 학습 (Mel Spectrogram)
    input_shape = (X_train.shape[1], X_train.shape[2])
    lstm_autoencoder_mel = create_lstm_autoencoder(input_shape)

    # 모델 학습 (Mel Spectrogram)
    lstm_autoencoder_mel.fit(
        X_train, X_train,
        epochs=100,
        batch_size=32,
        validation_data=(X_val, X_val),
        callbacks=[early_stopping]
    )

    # 학습된 Mel Spectrogram 모델 저장
    mel_model_filename = f"{equipment_type}_{equipment_id}_mel_lstm_autoencoder.h5"
    lstm_autoencoder_mel.save(mel_model_filename)
    print(f"Mel Spectrogram LSTM Autoencoder model saved as {mel_model_filename}\n")

for (equipment_type, equipment_id), group_data in train_df.groupby(['equipment', 'id']):
    print(f"Training models for Equipment: {equipment_type}, ID: {equipment_id}")
    # 3. MFCC와 Mel Spectrogram 결합하여 LSTM Autoencoder 모델 학습
    X_combined = np.array([np.concatenate([m, mel]) for m, mel in zip(group_data['mfcc_scaled'], group_data['mel_spectrogram_scaled'])])
    X_combined = X_combined.reshape((X_combined.shape[0], X_combined.shape[1], 1))  # (samples, time_steps, 1)

    # 데이터셋을 학습과 검증 세트로 나누기
    X_train, X_val = train_test_split(X_combined, test_size=0.2, random_state=42)

    # LSTM Autoencoder 모델 생성 및 학습 (Combined)
    input_shape = (X_train.shape[1], X_train.shape[2])
    lstm_autoencoder_combined = create_lstm_autoencoder(input_shape)

    # 모델 학습 (Combined)
    lstm_autoencoder_combined.fit(
        X_train, X_train,
        epochs=100,
        batch_size=32,
        validation_data=(X_val, X_val),
        callbacks=[early_stopping]
    )

    # 학습된 결합 모델 저장
    combined_model_filename = f"{equipment_type}_{equipment_id}_combined_lstm_autoencoder.h5"
    lstm_autoencoder_combined.save(combined_model_filename)
    print(f"Combined LSTM Autoencoder model saved as {combined_model_filename}\n")




import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

def create_lstm_autoencoder(input_shape):
    inputs = Input(shape=input_shape)

    x = LSTM(256, activation="tanh", recurrent_activation="sigmoid", return_sequences=True)(inputs)
    x = Dropout(0.2)(x)
    x = LSTM(128, activation="tanh", recurrent_activation="sigmoid", return_sequences=True)(x)
    x = Dropout(0.2)(x)
    x = LSTM(64, activation="tanh", recurrent_activation="sigmoid", return_sequences=False)(x)
    encoded = Dense(32, activation="relu")(x)

    x = RepeatVector(input_shape[0])(encoded)
    x = LSTM(64, activation="tanh", recurrent_activation="sigmoid", return_sequences=True)(x)
    x = Dropout(0.2)(x)
    x = LSTM(128, activation="tanh", recurrent_activation="sigmoid", return_sequences=True)(x)
    x = Dropout(0.2)(x)
    x = LSTM(256, activation="tanh", recurrent_activation="sigmoid", return_sequences=True)(x)
    decoded = TimeDistributed(Dense(input_shape[1]))(x)

    autoencoder = Model(inputs, decoded)
    autoencoder.compile(optimizer="adam", loss="mse")

    return autoencoder



'''--------------- 전이 학습 ------------- '''
# 1. 기본 모델 학습
# fan_0 데이터를 사용해 기본 모델 학습
fan_0_data = train_df[(train_df['equipment'] == 'fan') & (train_df['id'] == 0)]
X_fan_0 = np.array(fan_0_data['mfcc_scaled'].tolist())
X_fan_0 = X_fan_0.reshape((X_fan_0.shape[0], X_fan_0.shape[1], 1))

input_shape = (X_fan_0.shape[1], X_fan_0.shape[2])
base_model = create_lstm_autoencoder(input_shape)

# 기본 모델 학습
early_stopping = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
base_model.fit(X_fan_0, X_fan_0, epochs=150, batch_size=32, validation_split=0.2, callbacks=[early_stopping])

# 기본 모델 저장
base_model.save("fan_base_model_2.h5")

# 2. 각 ID별로 전이 학습 수행
for equipment_id in [0, 2, 4, 6]:  # 전이 학습할 fan ID 목록
    print(f"Fine-tuning for fan ID {equipment_id}")

    # 기본 모델 불러오기 및 일부 층 고정
    model = load_model("fan_base_model_2.h5")
    for layer in model.layers[:-3]:  # 하위 3개 층만 학습 가능하게 하고 나머지는 고정
        layer.trainable = False

    # ID별 데이터 준비
    id_data = train_df[(train_df['equipment'] == 'fan') & (train_df['id'] == equipment_id)]
    X_train_id = np.array(id_data['mfcc_scaled'].tolist())
    X_train_id = X_train_id.reshape((X_train_id.shape[0], X_train_id.shape[1], 1))

    # 전이 학습 수행
    model.fit(X_train_id, X_train_id, epochs=50, batch_size=32, validation_split=0.2, callbacks=[early_stopping])

    # ID별로 전이 학습된 모델 저장
    model.save(f"C:/Users/705-2/dataset_file/models/fan_id_{equipment_id}_fine_tuned_model_dropout2.h5")
    print(f"Fine-tuned model for fan ID {equipment_id} saved.")


import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, precision_recall_curve
import tensorflow as tf
import os

# 원하는 Precision 기준
desired_precision = 0.92

# 모델 파일이 저장된 절대 경로
base_path = r"C:\Users\705-2\dataset_file\models"

# 각 ID별 평가
for equipment_id in [2, 4, 6]:  # 평가할 fan ID 목록
    print(f"Evaluating model for Fan ID {equipment_id}")

    # 절대 경로로 파일 경로 작성
    model_path = f"{base_path}/fan_id_{equipment_id}_fine_tuned_model_dropout.h5"
    if not os.path.exists(model_path):
        print(f"Model file not found for Fan ID {equipment_id}")
        continue

    model = tf.keras.models.load_model(model_path)
    print(f"Loaded model for Fan ID {equipment_id}")

    # 테스트 데이터 준비 (해당 ID의 데이터만 필터링)
    test_data = test_df[(test_df['equipment'] == 'fan') & (test_df['id'] == equipment_id)]
    X_test = np.array(test_data['mfcc_scaled'].tolist())
    X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

    # Autoencoder를 통해 재구성 오류(MSE) 계산
    reconstructed = model.predict(X_test)
    mse = np.mean(np.power(X_test - reconstructed, 2), axis=(1, 2))

    # 실제 라벨 가져오기
    y_true = test_data['label'].values  # 정상 = 0, 비정상 = 1

    # Precision-Recall Curve를 사용하여 최적의 임계값 찾기
    precisions, recalls, thresholds = precision_recall_curve(y_true, mse)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)  # F1 스코어 계산
    valid_indices = np.where(precisions >= desired_precision)[0]

    if len(valid_indices) > 0:
        optimal_threshold = thresholds[valid_indices[np.argmax(f1_scores[valid_indices])]]
    else:
        optimal_threshold = thresholds[np.argmax(f1_scores)]

    print(f"Optimal Threshold for Fan ID {equipment_id}: {optimal_threshold}")

    # 최적 임계값을 사용한 이상 탐지
    y_pred = (mse > optimal_threshold).astype(int)

    # 평가 지표 계산
    auc_score = roc_auc_score(y_true, mse)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    # 결과 출력
    print(f"Evaluation Results for Fan ID {equipment_id}")
    print(f"AUC Score: {auc_score}")
    print(f"F1 Score: {f1}")
    print(f"Confusion Matrix:\n{cm}\n")

import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, precision_recall_curve
import tensorflow as tf
import os

# 모델 파일이 저장된 절대 경로
base_path = r"C:\Users\705-2\dataset_file\models"

# 각 ID별 평가
for equipment_id in [0, 2, 4, 6]:  # 평가할 fan ID 목록
    print(f"Evaluating model for Fan ID {equipment_id}")

    # 절대 경로로 파일 경로 작성
    model_path = f"{base_path}/fan_id_{equipment_id}_fine_tuned_model_dropout.h5"
    if not os.path.exists(model_path):
        print(f"Model file not found for Fan ID {equipment_id}")
        continue

    model = tf.keras.models.load_model(model_path)
    print(f"Loaded model for Fan ID {equipment_id}")

    # 테스트 데이터 준비 (해당 ID의 데이터만 필터링)
    test_data = test_df[(test_df['equipment'] == 'fan') & (test_df['id'] == equipment_id)]
    X_test = np.array(test_data['mfcc_scaled'].tolist())
    X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

    # Autoencoder를 통해 재구성 오류(MSE) 계산
    reconstructed = model.predict(X_test)
    mse = np.mean(np.power(X_test - reconstructed, 2), axis=(1, 2))

    # 실제 라벨 가져오기
    y_true = test_data['label'].values  # 정상 = 0, 비정상 = 1

    # Precision-Recall Curve를 사용하여 여러 Threshold 후보군 생성
    precisions, recalls, thresholds = precision_recall_curve(y_true, mse)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)  # F1 Score 계산

    # 최적의 F1 Score를 제공하는 Threshold 선택
    best_f1_score = 0
    best_threshold = thresholds[0]  # 초기값 설정

    for i, threshold in enumerate(thresholds):
        y_pred = (mse > threshold).astype(int)
        current_f1 = f1_score(y_true, y_pred)

        if current_f1 > best_f1_score:
            best_f1_score = current_f1
            best_threshold = threshold

    print(f"Best Threshold for Fan ID {equipment_id} based on F1 Score: {best_threshold}")

    # 최적 Threshold를 사용한 이상 탐지
    y_pred = (mse > best_threshold).astype(int)

    # 평가 지표 계산
    auc_score = roc_auc_score(y_true, mse)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    # 결과 출력
    print(f"Evaluation Results for Fan ID {equipment_id}")
    print(f"AUC Score: {auc_score}")
    print(f"F1 Score: {f1}")
    print(f"Confusion Matrix:\n{cm}\n")


''' ------------- fan_id_6을 이용한 fine_tuning ----------------'''
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

# fan ID 6 모델 불러오기
base_model_path = r"C:\Users\705-2\dataset_file\models\fan_id_6_fine_tuned_model.h5"
base_model = load_model(base_model_path)

# 상위 레이어 몇 개만 미세 조정 가능하게 설정
for layer in base_model.layers[:-3]:  # 하위 레이어 고정
    layer.trainable = False

# Fine-tuning을 위한 파라미터
epochs = 50
batch_size = 32
early_stopping = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

# Fine-tuning할 fan ID 데이터 목록
fine_tuning_ids = [0, 2, 4]

# 각 fan ID 데이터로 Fine-tuning 수행
for equipment_id in fine_tuning_ids:
    print(f"Fine-tuning for fan ID {equipment_id}")

    # fan ID 데이터 준비
    id_data = train_df[(train_df['equipment'] == 'fan') & (train_df['id'] == equipment_id)]
    X_train_id = np.array(id_data['mfcc_scaled'].tolist())
    X_train_id = X_train_id.reshape((X_train_id.shape[0], X_train_id.shape[1], 1))

    # Fine-tuning 수행
    base_model.fit(X_train_id, X_train_id,
                   epochs=epochs,
                   batch_size=batch_size,
                   validation_split=0.2,
                   callbacks=[early_stopping])

    # Fine-tuning 후 새로운 모델 저장
    fine_tuned_model_path = f"C:/Users/705-2/dataset_file/models/fan_id_{equipment_id}_fine_tuned_from_id_6_model.h5"
    base_model.save(fine_tuned_model_path)
    print(f"Fine-tuned model for fan ID {equipment_id} saved as {fine_tuned_model_path}")

from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, precision_recall_curve

# 평가할 fan ID 목록
evaluation_ids = [0, 2, 4]  # Fine-tuning한 모델들 포함

# 원하는 precision 기준
desired_precision = 0.8

for equipment_id in evaluation_ids:
    print(f"Evaluating fine-tuned model for Fan ID {equipment_id}")

    # Fine-tuned 모델 로드
    model_path = f"C:/Users/705-2/dataset_file/models/fan_id_{equipment_id}_fine_tuned_from_id_6_model.h5"
    model = load_model(model_path)

    # 평가용 데이터 준비
    test_data = test_df[(test_df['equipment'] == 'fan') & (test_df['id'] == equipment_id)]
    X_test = np.array(test_data['mfcc_scaled'].tolist())
    X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

    # 재구성 오류(MSE) 계산
    reconstructed = model.predict(X_test)
    mse = np.mean(np.power(X_test - reconstructed, 2), axis=(1, 2))

    # 실제 라벨 가져오기
    y_true = test_data['label'].values

    # Precision-Recall Curve를 사용하여 최적의 임계값 찾기
    precisions, recalls, thresholds = precision_recall_curve(y_true, mse)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
    valid_indices = np.where(precisions >= desired_precision)[0]

    if len(valid_indices) > 0:
        optimal_threshold = thresholds[valid_indices[np.argmax(f1_scores[valid_indices])]]
    else:
        optimal_threshold = thresholds[np.argmax(f1_scores)]

    print(f"Optimal threshold for Fan ID {equipment_id}: {optimal_threshold}")

    # 최적 임계값을 사용한 이상 탐지
    y_pred = (mse > optimal_threshold).astype(int)

    # 평가 지표 계산
    auc_score = roc_auc_score(y_true, mse)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    # 결과 출력
    print(f"Evaluation Results for Fan ID {equipment_id}")
    print(f"AUC Score: {auc_score}")
    print(f"F1 Score: {f1}")
    print(f"Confusion Matrix:\n{cm}\n")

def create_lstm_autoencoder(input_shape):
    inputs = Input(shape=input_shape)

    x = LSTM(256, activation="tanh", recurrent_activation="sigmoid", return_sequences=True)(inputs)
    x = Dropout(0.2)(x)
    x = LSTM(128, activation="tanh", recurrent_activation="sigmoid", return_sequences=True)(x)
    x = Dropout(0.2)(x)
    x = LSTM(64, activation="tanh", recurrent_activation="sigmoid", return_sequences=False)(x)
    encoded = Dense(32, activation="relu")(x)

    x = RepeatVector(input_shape[0])(encoded)
    x = LSTM(64, activation="tanh", recurrent_activation="sigmoid", return_sequences=True)(x)
    x = Dropout(0.2)(x)
    x = LSTM(128, activation="tanh", recurrent_activation="sigmoid", return_sequences=True)(x)
    x = Dropout(0.2)(x)
    x = LSTM(256, activation="tanh", recurrent_activation="sigmoid", return_sequences=True)(x)
    decoded = TimeDistributed(Dense(input_shape[1]))(x)

    autoencoder = Model(inputs, decoded)
    autoencoder.compile(optimizer="adam", loss="mse")

    return autoencoder

# 1. 기본 모델 학습
# fan_0 데이터를 사용해 기본 모델 학습
fan_0_data = train_df[(train_df['equipment'] == 'fan') & (train_df['id'] == 0)]
X_fan_0 = np.array(fan_0_data['mfcc_scaled'].tolist())
X_fan_0 = X_fan_0.reshape((X_fan_0.shape[0], X_fan_0.shape[1], 1))

input_shape = (X_fan_0.shape[1], X_fan_0.shape[2])
base_model = create_lstm_autoencoder(input_shape)

# 기본 모델 학습
early_stopping = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
base_model.fit(X_fan_0, X_fan_0, epochs=200, batch_size=32, validation_split=0.2, callbacks=[early_stopping])

# 기본 모델 저장
base_model.save('/content/drive/MyDrive/예비 프로젝트/텔레그램 모델/fan_base_model.keras')

# 2. 각 ID별로 전이 학습 수행
for equipment_id in [0, 2, 4, 6]:  # 전이 학습할 fan ID 목록
    print(f"Fine-tuning for fan ID {equipment_id}")

    # 기본 모델 불러오기 및 일부 층 고정
    model = load_model('/content/drive/MyDrive/예비 프로젝트/텔레그램 모델/fan_base_model.keras')
    for layer in model.layers[:-3]:  # 하위 3개 층만 학습 가능하게 하고 나머지는 고정
        layer.trainable = False

    # ID별 데이터 준비
    id_data = train_df[(train_df['equipment'] == 'fan') & (train_df['id'] == equipment_id)]
    X_train_id = np.array(id_data['mfcc_scaled'].tolist())
    X_train_id = X_train_id.reshape((X_train_id.shape[0], X_train_id.shape[1], 1))

    # 전이 학습 수행
    model.fit(X_train_id, X_train_id, epochs=50, batch_size=32, validation_split=0.2, callbacks=[early_stopping])

    # ID별로 전이 학습된 모델 저장
    model.save(f"/content/drive/MyDrive/예비 프로젝트/텔레그램 모델/fan_id_{equipment_id}_model_dropout.keras")
    print(f"Fine-tuned model for fan ID {equipment_id} saved.")